TEAM:


1.   SORONGON, CHARLES JUVANNE
2.   BONIEL, GERALD
3.   SISI, KENT JASPER
4.   LERIO, JARS CHRISTIAN



# CPU vs GPU Vector Addition using PyCUDA

### This activity demonstrates how vector addition is performed using both CPU and GPU execution. The CPU version uses NumPy as the baseline, while the GPU version uses PyCUDA to perform the same task using parallel processing. By comparing execution times, we can observe how GPU acceleration improves performance for larger datasets. The program also validates if both outputs are the same to ensure correctness. This follows the exact section-by-section procedure required in the activity sheet.

In [21]:
!nvidia-smi

Sat May  9 03:54:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P0             31W /   70W |     117MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [22]:
!pip install pycuda

# SECTION 1: Import Required Libraries

### This section imports all the necessary libraries needed for the program. NumPy is used for creating and processing arrays, time is used for measuring execution speed, and PyCUDA is used for running computations on the GPU. These libraries are required before starting both CPU and GPU implementations.

In [23]:
# numpy -> for arrays and vector operations
import numpy as np

# time -> for measuring execution time
import time

# pycuda -> for GPU programming
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule

# SECTION 2: Prepare Input Data

### In this section, two large arrays of floating-point numbers are created. These arrays serve as the input data for both CPU and GPU computations. The activity sheet requires using large input sizes to better observe the performance difference between sequential and parallel execution.

In [24]:
print("\n" + "="*60)
print("STEP 1: PREPARING INPUT DATA")
print("="*60)

# Number of elements
N = 1000000

# Create two random float arrays
a = np.random.randn(N).astype(np.float32)
b = np.random.randn(N).astype(np.float32)

print(f"Input size (N): {N:,}")
print("Two random arrays created successfully.\n")


STEP 1: PREPARING INPUT DATA
Input size (N): 1,000,000
Two random arrays created successfully.



# SECTION 3: CPU Implementation (Baseline)

### This section performs vector addition using the CPU through NumPy. This serves as the baseline version for comparison with the GPU implementation. The execution time is measured to determine how long the CPU takes to complete the operation.

In [25]:
print("="*60)
print("STEP 2: CPU IMPLEMENTATION (NUMPY)")
print("="*60)

# Start timer
start = time.time()

# CPU vector addition
c_cpu = a + b

# End timer
end = time.time()

# Compute execution time
cpu_time = end - start

print(f"CPU Execution Time: {cpu_time:.6f} seconds\n")

STEP 2: CPU IMPLEMENTATION (NUMPY)
CPU Execution Time: 0.001306 seconds



# SECTION 4: Define CUDA Kernel

### This section creates the CUDA kernel, which contains the instructions that the GPU will execute. Each GPU thread handles one element of the vector addition. The indexing formula helps identify which part of the data each thread should process.

In [26]:
print("="*60)
print("STEP 3: GPU IMPLEMENTATION (PYCUDA)")
print("="*60)

mod = SourceModule("""

__global__ void add_vectors(float *dest, float *a, float *b)
{
    // Get thread index
    int idx = threadIdx.x + blockIdx.x * blockDim.x;

    // Perform vector addition
    dest[idx] = a[idx] + b[idx];
}

""")

STEP 3: GPU IMPLEMENTATION (PYCUDA)


# SECTION 5: Allocate GPU Memory

### In this section, memory is allocated inside the GPU for the input arrays and the output result. After allocating memory, the data from the CPU is transferred to the GPU so that the GPU can perform the computation.

In [27]:
print("Allocating GPU memory...")

# Allocate memory inside GPU
a_gpu = cuda.mem_alloc(a.nbytes)
b_gpu = cuda.mem_alloc(b.nbytes)
c_gpu = cuda.mem_alloc(a.nbytes)

# Transfer data from CPU to GPU
cuda.memcpy_htod(a_gpu, a)
cuda.memcpy_htod(b_gpu, b)

print("Data transferred successfully to GPU.\n")

Allocating GPU memory...
Data transferred successfully to GPU.



# SECTION 6: Execute CUDA Kernel

### This section runs the GPU computation by calling the CUDA kernel. The block size determines how many threads are placed in one block, while the grid size determines how many blocks are needed to process all elements. This is where parallel execution happens.

In [28]:
print("="*60)
print("STEP 4: EXECUTING GPU KERNEL")
print("="*60)

# Get function from kernel
func = mod.get_function("add_vectors")

# Number of threads per block
block_size = 256

# Number of blocks needed
grid_size = (N + block_size - 1) // block_size

print(f"Block Size : {block_size}")
print(f"Grid Size  : {grid_size}\n")

# Start timer
start = time.time()

# Run kernel
func(
    c_gpu,
    a_gpu,
    b_gpu,
    block=(block_size, 1, 1),
    grid=(grid_size, 1)
)

# End timer
end = time.time()

# Compute GPU execution time
gpu_time = end - start

print(f"GPU Execution Time: {gpu_time:.6f} seconds\n")

STEP 4: EXECUTING GPU KERNEL
Block Size : 256
Grid Size  : 3907

GPU Execution Time: 0.000208 seconds



# SECTION 7: Retrieve GPU Result and Validate Output

### After GPU execution, the result must be copied back from the GPU to the CPU memory. The output is then compared with the CPU result using np.allclose() to verify if both implementations produce the same answer.

In [29]:

print("STEP 5: RESULT VALIDATION")
print("="*60)

# Create empty array for GPU result
c_result = np.empty_like(a)

# Copy result from GPU to CPU
cuda.memcpy_dtoh(c_result, c_gpu)

# Compare CPU and GPU results
match = np.allclose(c_cpu, c_result)

print(f"Do CPU and GPU results match? : {match}\n")

STEP 5: RESULT VALIDATION
Do CPU and GPU results match? : True



# SECTION 8: Performance Evaluation Table

### This section presents the execution times of both CPU and GPU implementations in a simple table format. This helps visualize the performance difference clearly and supports the required analysis in the activity.

In [30]:
print("="*60)
print("PERFORMANCE EVALUATION")
print("="*60)

print("+----------------------+--------------------------+")
print("| Implementation       | Execution Time (seconds) |")
print("+----------------------+--------------------------+")
print(f"| CPU (NumPy)          | {cpu_time:>22.6f} |")
print(f"| GPU (PyCUDA)         | {gpu_time:>22.6f} |")
print("+----------------------+--------------------------+")
print()

PERFORMANCE EVALUATION
+----------------------+--------------------------+
| Implementation       | Execution Time (seconds) |
+----------------------+--------------------------+
| CPU (NumPy)          |               0.001306 |
| GPU (PyCUDA)         |               0.000208 |
+----------------------+--------------------------+



# SECTION 9: Analysis

### This final section gives a short explanation of the observed results. It explains whether GPU execution was faster and why this happens. It also connects the results to the concept of parallel processing discussed in the lesson.

In [31]:
print("="*60)
print("ANALYSIS")
print("="*60)

if gpu_time < cpu_time:
    print("GPU execution is faster than CPU execution.")
    print("This shows the advantage of parallel processing.")
else:
    print("CPU execution is faster in this run.")
    print("This may happen because of GPU transfer overhead.")

print("\nFor larger datasets, GPU performance usually improves")
print("because thousands of GPU threads work in parallel.")
print("="*60)

ANALYSIS
GPU execution is faster than CPU execution.
This shows the advantage of parallel processing.

For larger datasets, GPU performance usually improves
because thousands of GPU threads work in parallel.


# PERFORMANCE OUTPUT

### This section shows:
### 1. CPU execution time
### 2. GPU execution time
### 3. Output validation (if results match)

In [32]:
print("\n" + "="*65)
print("                PERFORMANCE OUTPUT RESULTS")
print("="*65)

# -------------------------------
# CPU Execution Time
# -------------------------------
print("\n[1] CPU IMPLEMENTATION TIME")
print("-" * 65)
print(f"CPU Execution Time : {cpu_time:.6f} seconds")


# -------------------------------
# GPU Execution Time
# -------------------------------
print("\n[2] GPU IMPLEMENTATION TIME")
print("-" * 65)
print(f"GPU Execution Time : {gpu_time:.6f} seconds")


# -------------------------------
# Output Validation
# -------------------------------
print("\n[3] OUTPUT VALIDATION")
print("-" * 65)

if match:
    print("Validation Result  : SUCCESS")
    print("CPU and GPU outputs match correctly.")
else:
    print("Validation Result  : FAILED")
    print("CPU and GPU outputs do NOT match.")


# -------------------------------
# Final Comparison Table
# -------------------------------
print("\n" + "="*65)
print("               FINAL PERFORMANCE SUMMARY")
print("="*65)

print("+----------------------+--------------------------+")
print("| Implementation       | Execution Time (seconds) |")
print("+----------------------+--------------------------+")
print(f"| CPU (NumPy)          | {cpu_time:>22.6f} |")
print(f"| GPU (PyCUDA)         | {gpu_time:>22.6f} |")
print("+----------------------+--------------------------+")

print("\nOutput Validation Status :", "PASSED" if match else "FAILED")
print("="*65)


                PERFORMANCE OUTPUT RESULTS

[1] CPU IMPLEMENTATION TIME
-----------------------------------------------------------------
CPU Execution Time : 0.001306 seconds

[2] GPU IMPLEMENTATION TIME
-----------------------------------------------------------------
GPU Execution Time : 0.000208 seconds

[3] OUTPUT VALIDATION
-----------------------------------------------------------------
Validation Result  : SUCCESS
CPU and GPU outputs match correctly.

               FINAL PERFORMANCE SUMMARY
+----------------------+--------------------------+
| Implementation       | Execution Time (seconds) |
+----------------------+--------------------------+
| CPU (NumPy)          |               0.001306 |
| GPU (PyCUDA)         |               0.000208 |
+----------------------+--------------------------+

Output Validation Status : PASSED
